# Attention Is All You Need - 실습 코드 2: 논문 재현: WMT 번역 데이터로 Transformer 학습

- Tutorial ID: `expand-attention-is-all-you-need`
- Tutorial: Attention Is All You Need
- Section ID: `expand-attention-is-all-you-need-code-2`
- Section: 실습 코드 2: 논문 재현: WMT 번역 데이터로 Transformer 학습


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: 논문 재현: WMT 번역 데이터로 Transformer 학습
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 위치 정보가 벡터 회전/편향으로 attention score에 들어가는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ── Positional Encoding (논문 Section 3.5) ──
class PositionalEncoding(nn.Module):
        def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
                self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)  # 짝수 인덱스
        pe[:, 1::2] = torch.cos(position * div_term)  # 홀수 인덱스
        self.register_buffer('pe', pe.unsqueeze(0))

        def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


# ── Feed-Forward Network (논문 Section 3.3) ──
class PositionwiseFeedForward(nn.Module):
        def __init__(self, d_model=512, d_ff=2048, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
                self.dropout = nn.Dropout(dropout)

        def forward(self, x):
        return self.w_2(self.dropout(F.relu(self.w_1(x))))


# ── 전체 Encoder-Decoder (논문 Figure 1) ──
class EncoderLayer(nn.Module):
        def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
                self.dropout = nn.Dropout(dropout)

        def forward(self, x, mask=None):
        # Post-LN (논문 원래 방식)
                x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)[0]))
                x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


class DecoderLayer(nn.Module):
        def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
                self.masked_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
                self.dropout = nn.Dropout(dropout)

        def forward(self, x, enc_out, src_mask=None, tgt_mask=None):
                x = self.norm1(x + self.dropout(
            self.masked_attn(x, x, x, tgt_mask)[0]))
                x = self.norm2(x + self.dropout(
            self.cross_attn(x, enc_out, enc_out, src_mask)[0]))
                x = self.norm3(x + self.dropout(self.ffn(x)))
        return x


# Causal Mask 생성 (자기회귀 디코딩)
def generate_causal_mask(seq_len):
    """하삼각 마스크: 미래 토큰 참조 방지"""
        mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
    return mask  # (1, 1, seq, seq)


# 학습 루프
def train_transformer(model, train_loader, epochs=10, d_model=512):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9
    )
    scheduler = TransformerLRScheduler(optimizer, d_model, warmup_steps=4000)
    criterion = LabelSmoothing(vocab_size=37000, smoothing=0.1)
    
    model.train()
        for epoch in range(epochs):
                total_loss = 0
                for batch in train_loader:
            src, tgt = batch  # (batch, seq)
            tgt_input = tgt[:, :-1]
                        tgt_output = tgt[:, 1:]
            
            # Causal mask
                        tgt_mask = generate_causal_mask(tgt_input.size(1))
            
            # Forward
                        enc_out = model.encode(src)
                        dec_out = model.decode(tgt_input, enc_out, tgt_mask=tgt_mask)
                        logits = model.generator(dec_out)
            
            # Loss
                        loss = criterion(
                F.log_softmax(logits, dim=-1).contiguous().view(-1, 37000),
                tgt_output.contiguous().view(-1)
            )
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
        
                print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.2f}, lr={scheduler.steps ** -0.5:.6f}")